In [1]:
# import the necessary package
import numpy as np
import os
import cv2
import argparse
import time
import imutils
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import load_model
from imutils.video import VideoStream

In [2]:
def detect_and_predict_mask(frame, faceNet, maskNet):
    # grab the dimensions of the frame and then construct the blob from it:
    (h, w) = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), (104.0, 177.0, 123.0))

    # pass the blob through the network and obtain the face detections:
    faceNet.setInput(blob)
    detections= faceNet.forward()

    # initialize our list of faces, their corresponding locations, and the list of predictions from our face mask network:
    faces = []
    locs = []
    preds = []

    # loop over the detection:
    CONFIDENCE = 0.1
    for i in range(0, detections.shape[2]):
        confidence = detections[0, 0, i, 2]

        if confidence > CONFIDENCE:
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (startX, startY, endX, endY) = box.astype("int")
    
            (startX, startY) = (max(0, startX), max(0, startY))
            (endX, endY) = (min(w - 1, endX), min(h - 1, endY))
    
            face =frame[startY:endY, startX:endY]
            if face.shape[0] ==0 or face.shape[1] == 0:
                continue
                
            face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
            face = cv2.resize(face, (224,224))
            face = img_to_array(face)
            face = preprocess_input(face)
            face = np.expand_dims(face, axis=0)
    
            faces.append(face)
            locs.append((startX, startY, endX, endY))
    
        if len(faces)>0:
    
            faces = np.vstack(faces)
            #faces = np.array(faces, dtype="float32")
            preds = maskNet.predict(faces, batch_size = 32)
    
        return (locs, preds)

In [3]:
# construct the argument parser and parse the arguments
ap = argparse.ArgumentParser()
ap.add_argument("-f", "--face", type=str,
    default="face_detector",
    help="path to face detector model directory")
ap.add_argument("-m", "--model", type=str,
    default="mask_detector.model",
    help="path to trained face mask detector model")
ap.add_argument("-c", "--confidence", type=float, default=0.5,
    help="minimum probability to filter weak detections")
args = vars(ap.parse_args())

In [4]:
# load the serialized face detector model from disk

print("[INFO] loading face detector model...")
prototxtPath = os.path.join("face_detector", "deploy.prototxt")
weightsPath = os.path.join("face_detector", "res10_300x300_ssd_iter_140000.caffemodel")
faceNet = cv2.dnn.readNet(prototxtPath, weightsPath)

[INFO] loading face detector model...


In [5]:
# load the face mask detector model from disk

print("[INFO] loading face mask detector model...")
maskNet = load_model("model_mask_detector.h5")

[INFO] loading face mask detector model...


In [6]:
# initialise the video stream and allow the camera sensor to warm up:

print("[INFO] starting video stream...")
vs = VideoStream(src = 0).start()
time.sleep(2.0)

[INFO] starting video stream...


In [ ]:
# loop over the frames from the video stream:
while True:
    # grab the frame from the threaded video stream and resize it to have a maximum width of 400 pixels:

    frame = vs.read()
    if frame is None:
        break
    frame = imutils.resize(frame, width=400)

     # detect faces in the frame and determine if they are wearing a face mask or not:
    (locs, preds) = detect_and_predict_mask(frame, faceNet, maskNet)

     # loop over the detected face locations and their corresponding locations:
    for (box, pred) in zip(locs, preds):
        # unpack the bounding box and predictions:
        (startX, startY, endX, endY) = box
        (mask, withoutMask) = pred

        # determine the class label and color we'll use to draw the bounding box and text:
        label = "Mask" if mask > withoutMask else "No Mask"
        color = (0, 255, 0) if label == "Mask" else (0, 0, 255)

        # display the label and bounding box rectangle on the output frame:
     
        if (label == "Mask"):

            cv2.putText(frame, "Mask: You are Allowed", (startX, startY - 10), 
                         cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)
            cv2.rectangle(frame, (startX, startY), (endX, endY), color, 2)

        elif (label == "No Mask"):
            label = "No Mask: You are not allowed"
            cv2.putText(frame, label, (startX, startY - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)
            cv2.rectangle(frame, (startX, startY), (endX, endY), color, 2)
        
    # show the output frame
    cv2.imshow("Frame", frame)
    key = cv2.waitKey(1) & 0xFF

    # if the `q` key was pressed, break from the loop
    if key == ord("q"):
        break
# do a bit of cleanup
cv2.destroyAllWindows()

vs.stop()

    

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━

In [ ]:
import numpy as np
import os
import cv2
import argparse
import time
import imutils
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import load_model
from imutils.video import VideoStream

def detect_and_predict_mask(frame, faceNet, maskNet):
    # grab the dimensions of the frame and then construct the blob from it:
    (h, w) = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), (104.0, 177.0, 123.0))

    # pass the blob through the network and obtain the face detections:
    faceNet.setInput(blob)
    detections= faceNet.forward()

    # initialize our list of faces, their corresponding locations, and the list of predictions from our face mask network:
    faces = []
    locs = []
    preds = []

    # loop over the detection:
    CONFIDENCE = 0.1
    for i in range(0, detections.shape[2]):
        confidence = detections[0, 0, i, 2]

        if confidence > CONFIDENCE:
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (startX, startY, endX, endY) = box.astype("int")
    
            (startX, startY) = (max(0, startX), max(0, startY))
            (endX, endY) = (min(w - 1, endX), min(h - 1, endY))
    
            face =frame[startY:endY, startX:endY]
            if face.shape[0] ==0 or face.shape[1] == 0:
                continue
                
            face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
            face = cv2.resize(face, (224,224))
            face = img_to_array(face)
            face = preprocess_input(face)
            face = np.expand_dims(face, axis=0)
    
            faces.append(face)
            locs.append((startX, startY, endX, endY))
    
        if len(faces)>0:
    
            faces = np.vstack(faces)
            #faces = np.array(faces, dtype="float32")
            preds = maskNet.predict(faces, batch_size = 32)
    
        return (locs, preds)


# construct the argument parser and parse the arguments
ap = argparse.ArgumentParser()
ap.add_argument("-f", "--face", type=str,
    default="face_detector",
    help="path to face detector model directory")
ap.add_argument("-m", "--model", type=str,
    default="mask_detector.model",
    help="path to trained face mask detector model")
ap.add_argument("-c", "--confidence", type=float, default=0.5,
    help="minimum probability to filter weak detections")
args = vars(ap.parse_args())


# load the serialized face detector model from disk

print("[INFO] loading face detector model...")
prototxtPath = os.path.join("face_detector", "deploy.prototxt")
weightsPath = os.path.join("face_detector", "res10_300x300_ssd_iter_140000.caffemodel")
faceNet = cv2.dnn.readNet(prototxtPath, weightsPath)

# load the face mask detector model from disk

print("[INFO] loading face mask detector model...")
maskNet = load_model("model_mask_detector.h5")

# initialise the video stream and allow the camera sensor to warm up:

print("[INFO] starting video stream...")
vs = VideoStream(src = 0).start()
time.sleep(2.0)

# loop over the frames from the video stream:
while True:
    # grab the frame from the threaded video stream and resize it to have a maximum width of 400 pixels:

    frame = vs.read()
    if frame is None:
        break
    frame = imutils.resize(frame, width=400)

     # detect faces in the frame and determine if they are wearing a face mask or not:
    (locs, preds) = detect_and_predict_mask(frame, faceNet, maskNet)

     # loop over the detected face locations and their corresponding locations:
    for (box, pred) in zip(locs, preds):
        # unpack the bounding box and predictions:
        (startX, startY, endX, endY) = box
        (mask, withoutMask) = pred

        # determine the class label and color we'll use to draw the bounding box and text:
        label = "Mask" if mask > withoutMask else "No Mask"
        color = (0, 255, 0) if label == "Mask" else (0, 0, 255)

        # display the label and bounding box rectangle on the output frame:
     
        if (label == "Mask"):

            cv2.putText(frame, "Mask: You are Allowed", (startX, startY - 10), 
                         cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)
            cv2.rectangle(frame, (startX, startY), (endX, endY), color, 2)

        elif (label == "No Mask"):
            label = "No Mask: You are not allowed"
            cv2.putText(frame, label, (startX, startY - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)
            cv2.rectangle(frame, (startX, startY), (endX, endY), color, 2)
        
    # show the output frame
    cv2.imshow("Frame", frame)
    key = cv2.waitKey(1) & 0xFF

    # if the `q` key was pressed, break from the loop
    if key == ord("q"):
        break
# do a bit of cleanup
cv2.destroyAllWindows()

vs.stop()


